In [1]:
import os
from os import listdir
from os.path import join, isfile, isdir
import subprocess
import re

In [9]:
# define paths
data_path = "R:/JosephineTimm"
results_path = "//ieekf-fs1/home/timmjo/Animals/"
cluster_path = "/lustre/scratch/data/jtim_hpc-2PAnalysis"
login_string = "jtim_hpc@marvin.hpc.uni-bonn.de"

In [17]:
# get cohorts
print("The following cohorts are available:")
cohorts = [f for f in listdir(data_path) if isdir(join(data_path, f))]

# print cohorts
for item in range(len(cohorts)):
    print(f"{item}: {cohorts[item]}")

# pick cohort
command = int(input("Which cohort? >>>"))

The following cohorts are available:
0: Cohort_01_Training


Which cohort? >>> 0


In [16]:
# get animals
cohort_path = join(data_path,cohorts[command])
animals = [f for f in listdir(cohort_path) if isdir(join(cohort_path, f)) and f[0].isdigit()]
print(f"The followong animals are inside {cohorts[command]}:")
for item in animals:
    print(item)

The followong animals are inside Cohort_01_Training:
1_BECT849
2_OPI2661


In [20]:
# get folders on MARVIN
print("Retrieving information from the cluster...")
result = subprocess.run(["ssh", "marvin", "'ls'", f"'{cluster_path}'"],  capture_output = True)
entries = result.stdout.decode("UTF-8").split("\n")
cohort_entries = []
print("The following data folders can be found on the cluster:")
for entry in entries:
    if len(entry) >= 1:
        for animal in animals:
            if entry.split("_")[1] == animal.split("_")[1]:
                print(entry)
                cohort_entries.append(entry)

Retrieving information from the cluster...
The following data folders can be found on the cluster:
1_BECT849_S2_Baseline
1_BECT849_S4_Hab2
1_BECT849_S5_Scram
1_BECT849_S7_Predict
1_BECT849_S8_Predict
2_OPI2661_S1_Baseline
2_OPI2661_S2_Baseline
2_OPI2661_S3_Hab1
2_OPI2661_S4_Hab2
2_OPI2661_S5_Scram
2_OPI2661_S6_Scram
2_OPI2661_S7_Predict
2_OPI2661_S8_Predict
OPI_S1
OPI_S2
OPI_S3
OPI_S4
OPI_S5
OPI_S6
OPI_S7
OPI_S8



### Upload Functionality

In [25]:
upload = input("Do you want to upload data to MARVIN? (yes/no): ")
if upload.lower() == "yes":
    imaging_data = {}
    for animal in animals:
        animal_path = cohort_path + "/" + animal
        folders = [f for f in listdir(animal_path) if isdir(join(animal_path, f))]
        for item in folders:
            if re.match(r"^S\d+_", item):
                session_path = animal_path + '/' + item + "/Imaging_Data/"
                files = [f for f in listdir(session_path) if isfile(join(session_path, f))]
                for file in files:
                    if file[-3:] == "nd2":
                        imaging_data[f"{animal}_{item}"] = join(session_path, file)
            else: pass
                
    print("The following imaging data exists:")
    for item in imaging_data:
        print(f"{item} : {imaging_data[item]}")
    
    ### Make directories and copy    
    print("Uploading now...")
    for item in imaging_data:
        if item not in entries:
            subprocess.run(["ssh", "marvin", "'mkdir'", f"'{cluster_path}/{item}'"])
            print(f"Created directory for: {item}")
            subprocess.run(["scp", f"{imaging_data[item]}", f"{login_string}:{cluster_path}/{item}"])
            print(f"Copied nd2 file into {item}")
else: pass

Do you want to upload data to Marvin? (yes/no):  yes


The following imaging data exists:
1_BECT849_S2_Baseline : R:/JosephineTimm\Cohort_01_Training/1_BECT849/S2_Baseline/Imaging_Data/BECT849_250911_S2_Baseline.nd2
1_BECT849_S3_Hab1 : R:/JosephineTimm\Cohort_01_Training/1_BECT849/S3_Hab1/Imaging_Data/BECT849_250912_S3_Hab1.nd2
1_BECT849_S4_Hab2 : R:/JosephineTimm\Cohort_01_Training/1_BECT849/S4_Hab2/Imaging_Data/BECT849_250915_S4_Hab2.nd2
1_BECT849_S5_Scram : R:/JosephineTimm\Cohort_01_Training/1_BECT849/S5_Scram/Imaging_Data/BEC-T849_250916_S5_Scram.nd2
1_BECT849_S7_Predict : R:/JosephineTimm\Cohort_01_Training/1_BECT849/S7_Predict/Imaging_Data/BEC-T849_250922_S7_Predict.nd2
1_BECT849_S8_Predict : R:/JosephineTimm\Cohort_01_Training/1_BECT849/S8_Predict/Imaging_Data/BEC-T849_250923_S8_Predict.nd2
2_OPI2661_S1_Baseline : R:/JosephineTimm\Cohort_01_Training/2_OPI2661/S1_Baseline/Imaging_Data/OPI-2661_250929_S1_Baseline.nd2
2_OPI2661_S2_Baseline : R:/JosephineTimm\Cohort_01_Training/2_OPI2661/S2_Baseline/Imaging_Data/OPI-2661_250929_S2_Base

KeyboardInterrupt: 

### Download Functionality

In [12]:
download = input("Do you want to download results from MARVIN? (yes/no): ")
if download.lower() == "yes":
    for entry in entries:
        parts = entry.split("_")
        try:
            entry_folder = join(results_path,cohorts[command]) + "/" + f"{parts[0]}_{parts[1]}/{parts[2]}_{parts[3]}/Imaging_Data"
            entry_folder_contents = [f for f in listdir(entry_folder) if isdir(join(entry_folder, f))]
            if not "suite2p" in entry_folder_contents:
                subprocess.check_output(["scp", "-r", f"{login_string}:{cluster_path}/{entry}/suite2p", f"{entry_folder}"])
                print(f"Copied suite2p data into {entry_folder}")
            else:
                print(f"Suite2p data already exists for entry: {entry}")
        except subprocess.CalledProcessError:
            print(f"No suite2p results for entry: {entry}")
        except:
            print(f"No entry folder for entry: {entry}")
else: pass

No suite2p results for entry: 1_BECT849_S2_Baseline
No suite2p results for entry: 1_BECT849_S3_Hab1
No suite2p results for entry: 1_BECT849_S4_Hab2
No suite2p results for entry: 1_BECT849_S5_Scram
No suite2p results for entry: 1_BECT849_S7_Predict
No suite2p results for entry: 1_BECT849_S8_Predict
Suite2p data already exists for entry: 2_OPI2661_S1_Baseline
Copied suite2p data into //ieekf-fs1/home/timmjo/Animals/Cohort_01_Training/2_OPI2661/S2_Baseline/Imaging_Data
No suite2p results for entry: 2_OPI2661_S3_Hab1
No suite2p results for entry: 2_OPI2661_S4_Hab2
Suite2p data already exists for entry: 2_OPI2661_S5_Scram
Suite2p data already exists for entry: 2_OPI2661_S6_Scram
Suite2p data already exists for entry: 2_OPI2661_S7_Predict
No suite2p results for entry: 2_OPI2661_S8_Predict
No entry folder for entry: OPI_S1
No entry folder for entry: OPI_S2
No entry folder for entry: OPI_S3
No entry folder for entry: OPI_S4
No entry folder for entry: OPI_S5
No entry folder for entry: OPI_S6
No

### Delete Functionality

In [ ]:
delete = input("Do you want to delete files from MARVIN? (yes/no): ")
if delete.lower() == "yes":
    for entry in entries
else: pass